# install

In [1]:
# OpenAI & tokenizer
!pip install openai tiktoken

# Agent packages & tools
!pip install langchain langchain-experimental langchain-openai langgraph smolagents tavily-python loguru duckduckgo-search

# RAG packages & tools
!pip install lancedb sentence-transformers tantivy beautifulsoup4 ragas

INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This

# import

In [2]:
import os
import io
from getpass import getpass
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

from loguru import logger
from tavily import TavilyClient

import kagglehub

from google.colab import files

import torch.nn as nn
import torchvision
import torch
import torchvision.transforms as transforms

import numpy as np
from PIL import Image

from langgraph.graph import StateGraph, END
from typing import List, TypedDict

# EfficintNet-B1 Classifier

## Get .ckpt

In [3]:
! pip install kaggle

In [4]:
files.upload() # kaggle.json
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [5]:
# !kaggle kernels output stefantiberius/assignment9-efficient-net-b1 -p /img-model

Kernel log downloaded to /img-model/assignment9-efficient-net-b1.log 


In [10]:
# !kaggle kernels output stefantiberius/assignment9-efficient-net-b1 -p /img-model

Kernel log downloaded to /img-model/assignment9-efficient-net-b1.log 


## Class map

In [6]:
path = kagglehub.dataset_download("yudhaislamisulistya/plants-type-datasets")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'plants-type-datasets' dataset.
Path to dataset files: /kaggle/input/plants-type-datasets


In [7]:
train_root_dir = f'{path}/split_ttv_dataset_type_of_plants/Train_Set_Folder'
classes = sorted(os.listdir(train_root_dir))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
print(class_to_idx)


{'aloevera': 0, 'banana': 1, 'bilimbi': 2, 'cantaloupe': 3, 'cassava': 4, 'coconut': 5, 'corn': 6, 'cucumber': 7, 'curcuma': 8, 'eggplant': 9, 'galangal': 10, 'ginger': 11, 'guava': 12, 'kale': 13, 'longbeans': 14, 'mango': 15, 'melon': 16, 'orange': 17, 'paddy': 18, 'papaya': 19, 'peper chili': 20, 'pineapple': 21, 'pomelo': 22, 'shallot': 23, 'soybeans': 24, 'spinach': 25, 'sweet potatoes': 26, 'tobacco': 27, 'waterapple': 28, 'watermelon': 29}


## Model

In [8]:
class ImgRecognitionModel(nn.Module):

    def __init__(self, class_to_idx: dict):
        super().__init__()

        self.class_to_idx = class_to_idx
        self.idx_to_class = {v: k for k, v in class_to_idx.items()}

        num_classes = len(class_to_idx)

        self.model = torchvision.models.efficientnet_b1(weights=None)
        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


## Load model

In [11]:
uploaded = files.upload()

CKPT_FILE_PATH = None
for filename in uploaded.keys():
    CKPT_FILE_PATH = filename
    print(f"Using uploaded file: {filename}")
    break # Assumes only one file is uploaded

Saving best-efficientnet-b1-model.ckpt to best-efficientnet-b1-model.ckpt
Using uploaded file: best-efficientnet-b1-model.ckpt


In [12]:
imgRecognitionModel = ImgRecognitionModel(class_to_idx=class_to_idx)

checkpoint = torch.load(
    CKPT_FILE_PATH,
    map_location=torch.device('cpu')
    )

imgRecognitionModel.load_state_dict(checkpoint['state_dict'])
imgRecognitionModel.eval()


ImgRecognitionModel(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (s

## Predict

In [13]:
def predict(img: Image.Image, model: ImgRecognitionModel) -> str:

    trf = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    with torch.no_grad():
        X = trf(img).unsqueeze(0)
        y = model(X)
        y_hat = y.argmax(dim=1)

    return model.idx_to_class[y_hat.item()]


# Agent

## LLM model

In [14]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

LANGGRAPH_MODEL = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    temperature=0,
)


Enter your OpenAI API key: ··········


## Tavity

In [15]:
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key: ")

TAVILY_CLIENT = TavilyClient()

Enter your Tavily API key: ··········


## PlantCareCard

In [16]:
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum

class LightingCondition(str, Enum):
    FULL_SUN = "full_sun"
    PARTIAL_SHADE = "partial_shade"
    SHADE = "shade"
    INDIRECT_LIGHT = "indirect_light"

class WateringFrequency(str, Enum):
    DAILY = "daily"
    EVERY_2_3_DAYS = "every_2_3_days"
    WEEKLY = "weekly"
    BIWEEKLY = "biweekly"
    MONTHLY = "monthly"

class Difficulty(str, Enum):
    BEGINNER = "beginner"
    INTERMEDIATE = "intermediate"
    ADVANCED = "advanced"

class Season(str, Enum):
    SPRING = "spring"
    SUMMER = "summer"
    FALL = "fall"
    WINTER = "winter"

class PlantCareCard(BaseModel):
    """Structured plant care information card."""

    # Basic identification
    common_name: str = Field(description="Common name of the plant")
    latin_name: str = Field(description="Scientific/botanical Latin name")
    family: Optional[str] = Field(None, description="Plant family (e.g., Liliaceae)")

    # Growing conditions
    outdoors: bool = Field(description="Can be grown outdoors")
    indoors: bool = Field(description="Can be grown indoors")
    lighting_conditions: List[LightingCondition] = Field(
        description="Suitable lighting conditions"
    )

    # Care requirements
    watering_frequency: WateringFrequency = Field(
        description="How often to water"
    )
    humidity_level: str = Field(
        description="Preferred humidity (e.g., 'low', 'moderate', 'high')"
    )
    temperature_range_celsius: str = Field(
        description="Optimal temperature range (e.g., '18-24°C')"
    )
    soil_type: str = Field(
        description="Preferred soil type (e.g., 'well-draining sandy loam')"
    )

    # Maintenance
    difficulty_level: Difficulty = Field(description="Care difficulty level")
    fertilizer_needs: str = Field(
        description="Fertilization requirements (e.g., 'monthly during growing season')"
    )
    pruning_required: bool = Field(description="Whether regular pruning is needed")

    # Growth characteristics
    mature_height_cm: Optional[str] = Field(
        None, description="Expected mature height range"
    )
    growth_rate: str = Field(
        description="Growth speed (e.g., 'fast', 'moderate', 'slow')"
    )

    # Seasonal information
    blooming_season: Optional[List[Season]] = Field(
        None, description="When the plant blooms"
    )
    planting_season: Optional[List[Season]] = Field(
        None, description="Best time to plant"
    )

    # Additional care
    toxicity: str = Field(
        description="Toxicity info (e.g., 'toxic to pets', 'safe', 'mildly toxic')"
    )
    common_pests: List[str] = Field(
        default_factory=list,
        description="Common pests to watch for"
    )
    common_diseases: List[str] = Field(
        default_factory=list,
        description="Common diseases"
    )

    # Quick tips
    care_tips: List[str] = Field(
        description="3-5 practical care tips",
        min_length=3,
        max_length=5
    )

    class Config:
        use_enum_values = True

## AgentState

In [17]:
class AgentState(TypedDict):
    plant: str
    plant_care_card: PlantCareCard
    validation_feedback: Optional[str]
    content: List[str]
    revision_number: int
    max_revisions: int


## Prompts

### RESEARCH_PROMPT

In [18]:
RESEARCH_PROMPT = """You are a botanical researcher gathering plant care information.

Generate 3 targeted search queries to find:
1. Basic botanical info (scientific name, family, native habitat)
2. Core care requirements (light, water, soil, temperature, humidity)
3. Common problems (pests, diseases, toxicity)
"""

### CARD_GENERATION_PROMPT

In [19]:
CARD_GENERATION_PROMPT = """You are a world-class botanist creating a structured plant care reference card.

Based on the research provided, generate a complete PlantCareCard with:
- Accurate botanical nomenclature
- Specific, measurable care instructions (e.g., "water weekly" not "water regularly")
- Appropriate difficulty level for typical home gardeners
- 3-5 most critical care tips
- Complete pest/disease/toxicity information

Be conservative and precise. If information is uncertain, use general safe defaults.
"""


### VALIDATION_PROMPT

In [20]:
VALIDATION_PROMPT = """You are a horticultural expert reviewing a plant care card for accuracy and completeness.

Check for:
1. **Accuracy**: Botanical name correct? Care requirements safe and appropriate?
2. **Completeness**: Any critical care info missing? (light/water/soil/temp/toxicity)
3. **Specificity**: Are instructions actionable? (avoid vague terms like "moderate")
4. **Consistency**: Do recommendations align? (e.g., "full sun" + "indoors" = possible issue)

Provide:
- List of specific errors to fix
- Missing critical information to add
- Recommendations for improvement

If the card is accurate and complete, respond with: "APPROVED - No changes needed"
"""


## Nodes

In [21]:
class Queries(BaseModel):
    queries: List[str]

def research_node(state: AgentState):
    """
    Agent that plans and performs search queries based on the plant name using Tavily.
    """
    logger.info(f"RESEARCHER: Planning search queries...")
    messages = [
        SystemMessage(content=RESEARCH_PROMPT),
        HumanMessage(content=f"Generate search queries for gathering information about the plant: {state['plant']}")
    ]
    queries = LANGGRAPH_MODEL.with_structured_output(Queries).invoke(messages)

    logger.info(f"RESEARCHER: Search queries planned: {queries.queries}. Searching...")
    content = state.get('content', [])

    for i, q in enumerate(queries.queries, 1):
        logger.info(f"  Query {i}/{len(queries.queries)}: {q}")
        response = TAVILY_CLIENT.search(query=q, max_results=2)

        for r in response['results']:
            source_url = r.get('url', 'N/A')
            content.append(f"Source: {source_url}\n{r['content']}")

    logger.info(f"RESEARCHER: Search completed. Results: {content}")
    return {'content': content}


def generate_card_node(state: AgentState):
    """
    Agent that generates an article draft based on the plant name and the article outline plan.
    """
    logger.info(f"GENERATOR: Generating article draft...")
    content = "\n\n".join(state.get('content', []))

    validation_feedback = state.get('validation_feedback', '')
    feedback_section = ""
    if validation_feedback:
        feedback_section = f"\n\n**CRITICAL - Address these issues:**\n{validation_feedback}"

    messages = [
        SystemMessage(
            content=CARD_GENERATION_PROMPT
        ),
        HumanMessage(
            content=f"""Create a PlantCareCard for: **{state['plant']}**

            **Research Content:**
            {content}
            {feedback_section}
            """
        )
    ]
    plant_care_card = LANGGRAPH_MODEL.with_structured_output(PlantCareCard).invoke(messages)
    logger.info(f"CARD GENERATOR: {plant_care_card}")
    return {
        'plant_care_card': plant_care_card,
        'revision_number': state.get('revision_number', 0) + 1
    }


def validate_node(state: AgentState):
    """
    Agent that validates the PlantCareCard for accuracy and completeness.
    """
    logger.info("VALIDATOR: Reviewing PlantCareCard...")

    card = state['plant_care_card']

    messages = [
        SystemMessage(content=VALIDATION_PROMPT),
        HumanMessage(content=f"""Review this PlantCareCard for **{state['plant']}**:

{card.model_dump_json(indent=2)}

Evaluate for:
1. Botanical accuracy (scientific name, family)
2. Care completeness (all essential fields filled)
3. Specificity (measurable instructions, not vague)
4. Safety (toxicity info present and accurate)

Provide specific issues to fix OR respond "APPROVED" if card is complete and accurate.""")
    ]

    response = LANGGRAPH_MODEL.invoke(messages)

    # Check for approval
    if "APPROVED" in response.content.upper():
        logger.info("VALIDATOR: PlantCareCard approved ✓")
        return {'validation_feedback': None}  # None = approved

    logger.info("VALIDATOR: Issues found, requesting revision")
    return {'validation_feedback': response.content}

def should_continue(state: AgentState) -> str:
    """
    Determines next step: re-research, revise, or end.
    """
    # If approved (validation_feedback is None), end
    if state.get('validation_feedback') is None:
        logger.info("ROUTER: PlantCareCard approved. Ending.")
        return END

    # If max revisions reached, end
    if state['revision_number'] >= state['max_revisions']:
        logger.info(f"ROUTER: Max revisions ({state['max_revisions']}) reached. Ending.")
        return END

    # Check if validation mentions missing information (need re-research)
    feedback = state.get('validation_feedback', '').lower()
    missing_keywords = ["missing", "lack", "insufficient", "no information", "not found"]

    if any(keyword in feedback for keyword in missing_keywords):
        logger.info("ROUTER: Missing info detected. Re-researching...")
        return "research"  # Loop back to research

    # Otherwise, just revise the card
    logger.info(f"ROUTER: Revising card (revision {state['revision_number']}/{state['max_revisions']})...")
    return "revise"

## Graph

In [22]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node("researcher", research_node)
graph_builder.add_node("generator", generate_card_node)
graph_builder.add_node("validator", validate_node)
graph_builder.add_node("should_continue", should_continue)

graph_builder.set_entry_point("researcher")

graph_builder.add_edge("researcher", "generator")
graph_builder.add_edge("generator", "validator")

graph_builder.add_conditional_edges(
    "validator",
    should_continue,
    {
        "revise": "generator",
        "research": "researcher",
        END: END
    }
)

graph = graph_builder.compile()


# Botanist

In [23]:
thread = {
    "configurable": {"thread_id": "1"}
}

uploaded = files.upload()

img_bytes = list(uploaded.values())[0]
img = Image.open(io.BytesIO(img_bytes))

plant = predict(img, imgRecognitionModel)
print(plant)

plant_card = graph.invoke({
    'plant': plant,
    'max_revisions': 2,
    'revision_number': 1
}, thread)

plant_card

Saving aug_0_1162.jpg to aug_0_1162.jpg


2025-11-17 18:40:46.766 | INFO     | __main__:research_node:8 - RESEARCHER: Planning search queries...


coconut


2025-11-17 18:41:00.144 | INFO     | __main__:research_node:15 - RESEARCHER: Search queries planned: ['Cocos nucifera botanical information scientific name family Arecaceae native range habitat distribution description', 'Coconut palm (Cocos nucifera) care requirements light water soil temperature humidity irrigation fertilization soil pH drainage pot vs ground cultivation guidelines', 'Cocos nucifera common pests diseases red palm weevil rhinoceros beetle lethal yellowing bud rot Phytophthora symptoms diagnosis management control toxicity to humans and pets safety of coconuts']. Searching...
2025-11-17 18:41:00.145 | INFO     | __main__:research_node:19 -   Query 1/3: Cocos nucifera botanical information scientific name family Arecaceae native range habitat distribution description
2025-11-17 18:41:03.101 | INFO     | __main__:research_node:19 -   Query 2/3: Coconut palm (Cocos nucifera) care requirements light water soil temperature humidity irrigation fertilization soil pH drainage 

{'plant': 'coconut',
 'plant_care_card': PlantCareCard(common_name='Coconut (coconut palm)', latin_name='Cocos nucifera L.', family='Arecaceae', outdoors=True, indoors=True, lighting_conditions=['full_sun', 'indirect_light'], watering_frequency='every_2_3_days', humidity_level='high (preferably >60%)', temperature_range_celsius='21-32°C (avoid prolonged temps <18°C)', soil_type='Well-draining sandy loam; mix potting soil with coarse sand/perlite and some organic matter (e.g., 50% mineral, 50% organic) for containers', difficulty_level='intermediate', fertilizer_needs='Fertilize monthly during the warm growing season with a palm-specific or balanced fertilizer higher in potassium; supplement with magnesium and trace elements if deficiency appears', pruning_required=True, mature_height_cm='1000-3000 cm', growth_rate='fast (in warm, sunny, moist conditions)', blooming_season=['spring', 'summer', 'fall', 'winter'], planting_season=['spring', 'summer'], toxicity='Edible; generally not toxic

In [24]:
print(plant_card['plant_care_card'].model_dump_json(indent=2))


{
  "common_name": "Coconut (coconut palm)",
  "latin_name": "Cocos nucifera L.",
  "family": "Arecaceae",
  "outdoors": true,
  "indoors": true,
  "lighting_conditions": [
    "full_sun",
    "indirect_light"
  ],
  "watering_frequency": "every_2_3_days",
  "humidity_level": "high (preferably >60%)",
  "temperature_range_celsius": "21-32°C (avoid prolonged temps <18°C)",
  "soil_type": "Well-draining sandy loam; mix potting soil with coarse sand/perlite and some organic matter (e.g., 50% mineral, 50% organic) for containers",
  "difficulty_level": "intermediate",
  "fertilizer_needs": "Fertilize monthly during the warm growing season with a palm-specific or balanced fertilizer higher in potassium; supplement with magnesium and trace elements if deficiency appears",
  "pruning_required": true,
  "mature_height_cm": "1000-3000 cm",
  "growth_rate": "fast (in warm, sunny, moist conditions)",
  "blooming_season": [
    "spring",
    "summer",
    "fall",
    "winter"
  ],
  "planting_seas